In [16]:
%pip install holidays
%pip install tensorflow
%pip install scikit-learn
%pip install pandas
%pip install numpy
%pip install lets-plot

In [17]:
import pandas as pd
import numpy as np
import holidays

# Make numpy values easier to read.
np.set_printoptions(precision=3, suppress=True)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from lets_plot import *
LetsPlot.setup_html()

In [18]:
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv")

bikes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   dteday        112475 non-null  object 
 1   hr            112475 non-null  float64
 2   casual        112475 non-null  int64  
 3   registered    112475 non-null  int64  
 4   temp_c        112475 non-null  float64
 5   feels_like_c  112475 non-null  float64
 6   hum           112475 non-null  float64
 7   windspeed     112475 non-null  float64
 8   weathersit    112475 non-null  int64  
 9   season        112475 non-null  int64  
 10  holiday       112475 non-null  int64  
 11  workingday    112475 non-null  int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 10.3+ MB


## Make some features

In [19]:
def create_features(df):
    # holidays
    df['dteday'] = pd.to_datetime(df['dteday'])
    dc_holidays = holidays.US(state='DC')
    df['holiday_name'] = df['dteday'].apply(lambda x: dc_holidays.get(x))
    holiday_features = pd.get_dummies(df['holiday_name'], dtype=int)
    df = pd.concat([df, holiday_features], axis=1)

    # hour of day cycles
    df["hour_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

    #  year
    df['year'] = df['dteday'].dt.year

    # month cycle
    df['month'] = df['dteday'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # day of year cycle
    df['day_of_year'] = df['dteday'].dt.dayofyear
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # day cycle
    df['day_of_week'] = df['dteday'].dt.dayofweek.astype(int)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    #  season cycle
    df['season_sin'] = np.sin(2 * np.pi * df['season'] / 4)
    df['season_cos'] = np.cos(2 * np.pi * df['season'] / 4)

    # temperature
    df['temp_diff'] = df['feels_like_c'] - df['temp_c']

    # day type
    df['day_type'] = df.apply(lambda row: 'working_day' if row['holiday'] == 0 and row['workingday'] == 1 else ('weekend' if row['workingday'] == 0 and row['holiday'] == 0 else 'holiday_weekend' if row['holiday'] == 1 and row['workingday'] == 0 else 'holiday_weekday'), axis=1)
    day_type_features = pd.get_dummies(df['day_type'], dtype=int)
    df = pd.concat([df, day_type_features], axis=1)


    return df.drop(columns=['holiday_name', 'month', 'day_of_week', 'temp_c', 'season', 'day_of_year', 'day_type', 'dteday'])

In [20]:
df = create_features(bikes)

df.info()

df.describe().transpose()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112475 entries, 0 to 112474
Data columns (total 45 columns):
 #   Column                                                   Non-Null Count   Dtype  
---  ------                                                   --------------   -----  
 0   hr                                                       112475 non-null  float64
 1   casual                                                   112475 non-null  int64  
 2   registered                                               112475 non-null  int64  
 3   feels_like_c                                             112475 non-null  float64
 4   hum                                                      112475 non-null  float64
 5   windspeed                                                112475 non-null  float64
 6   weathersit                                               112475 non-null  int64  
 7   holiday                                                  112475 non-null  int64  
 8   workingday    

,count,mean,std,min,25%,50%,75%,max
hr,112475.0,11.501098,6.921864,0.000000,6.000000,1.200000e+01,1.800000e+01,23.000000
casual,112475.0,90.434612,128.655621,0.000000,7.000000,3.600000e+01,1.220000e+02,1244.000000
registered,112475.0,249.193625,258.267544,0.000000,48.000000,1.800000e+02,3.600000e+02,1702.000000
feels_like_c,112475.0,14.659325,11.428324,-24.000000,5.400000,1.600000e+01,2.350000e+01,49.600000
hum,112475.0,0.636624,0.190328,0.088900,0.484100,6.409000e-01,7.988000e-01,1.000000
windspeed,112475.0,13.100614,7.857600,0.000000,7.700000,1.220000e+01,1.750000e+01,69.800000
weathersit,112475.0,1.405441,0.683450,1.000000,1.000000,1.000000e+00,2.000000e+00,4.000000
holiday,112475.0,0.030300,0.171412,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000
workingday,112475.0,0.684312,0.464791,0.000000,0.000000,1.000000e+00,1.000000e+00,1.000000
Christmas Day,112475.0,0.002561,0.050537,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000


## Train

In [21]:
X = df.drop(columns=["casual", "registered"])
y = df[["casual", "registered"]]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler_x = MinMaxScaler()
scaler_y_casual = MinMaxScaler()
scaler_y_registered = MinMaxScaler()


scaler_x.fit(X_train)
scaler_y_casual.fit(y_train[["casual"]])
scaler_y_registered.fit(y_train[["registered"]])


X_train_s = scaler_x.transform(X_train)
X_test_s = scaler_x.transform(X_test)

# For y, we transform columns individually and combine them
y_train_s = np.hstack([
    scaler_y_casual.transform(y_train[["casual"]]),
    scaler_y_registered.transform(y_train[["registered"]])
])

y_test_s = np.hstack([
    scaler_y_casual.transform(y_test[["casual"]]),
    scaler_y_registered.transform(y_test[["registered"]])
])

In [38]:
model = Sequential()
model.add(Dense(128, input_dim=X_train_s.shape[1], activation='relu'))
model.add(Dropout(.25))
model.add(Dense(256, activation='relu'))
model.add(Dropout(.5))
model.add(Dense(64, activation='leaky_relu'))
model.add(Dropout(.25))

model.add(Dense(2, activation='linear'))

In [39]:
opt = keras.optimizers.Adam(learning_rate=0.0005)
model.compile(loss="mean_squared_error", optimizer=opt, metrics=['mse'])

In [40]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_mse', patience=30)

history = model.fit(
    X_train_s, 
    y_train_s, 
    epochs=2000, 
    validation_split=0.5, 
    batch_size=256,       
    callbacks=[early_stop], 
    shuffle=True          
)

hist = pd.DataFrame(history.history)

Epoch 1/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - loss: 0.0219 - mse: 0.0219 - val_loss: 0.0106 - val_mse: 0.0106
Epoch 2/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0110 - mse: 0.0110 - val_loss: 0.0083 - val_mse: 0.0083
Epoch 3/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0089 - mse: 0.0089 - val_loss: 0.0067 - val_mse: 0.0067
Epoch 4/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0076 - mse: 0.0076 - val_loss: 0.0057 - val_mse: 0.0057
Epoch 5/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0066 - mse: 0.0066 - val_loss: 0.0050 - val_mse: 0.0050
Epoch 6/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0059 - mse: 0.0059 - val_loss: 0.0043 - val_mse: 0.0043
Epoch 7/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0053 - mse: 0.0053 - val_loss: 0.0038 - val_mse: 0.0038
Epoch 8/2000
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0049 - mse: 0.0049 - val_loss: 0.0037 - val_mse: 0.0037
Epoch 9/2000
176/176 ━━━━━━━━━━━━━━━━━━

In [41]:
# 1. Create the DataFrame fresh from the history object
hist = pd.DataFrame(history.history)

# 2. Explicitly add an 'epoch' column (starting from 1)
hist['epoch'] = range(1, len(hist) + 1)

def plot_history():
    # 3. Use 'epoch' instead of 'index' for the melt
    plot_data = hist.melt(id_vars=['epoch'], 
                          value_vars=['mse', 'val_mse'], 
                          var_name='Type', 
                          value_name='MSE')

    # 4. Use 'epoch' for the x-axis
    p = ggplot(plot_data, aes(x='epoch', y='MSE', color='Type')) + \
        geom_line(size=1) + \
        labs(title="Model Training History", x="Epoch", y="Mean Square Error") + \
        theme_minimal() + \
        scale_color_discrete(labels=['Train Error', 'Val Error'])
    
    return p

# Display the plot
plot_history().show()

In [42]:
preds = model.predict(X_test_s)

703/703 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [43]:
pred_casual_s = preds[:, 0].reshape(-1, 1)
pred_registered_s = preds[:, 1].reshape(-1, 1)


final_casual = scaler_y_casual.inverse_transform(pred_casual_s)
final_registered = scaler_y_registered.inverse_transform(pred_registered_s)


results = pd.DataFrame({
    "predicted_casual": final_casual.flatten(),
    "predicted_registered": final_registered.flatten()
})

In [44]:
import joblib

# 1. Save the Keras model
# The '.keras' extension is the modern standard for TensorFlow/Keras
model.save("bike_rental_model.keras") 

# 2. Save the fitted scalers
joblib.dump(scaler_x, "scaler_x.pkl")
joblib.dump(scaler_y_casual, "scaler_y_casual.pkl")
joblib.dump(scaler_y_registered, "scaler_y_registered.pkl")

print("Model and scalers successfully saved!")

Model and scalers successfully saved!
